In [0]:
%pip install faker polars

In [0]:
import random
import uuid
import json
from datetime import datetime, timedelta
import polars as pl
from faker import Faker

Shared Pools (Keeps IDs consistent across tables)

In [0]:
from config import CUSTOMER_IDS, PRODUCT_IDS, CHANNELS, PAYMENT_METHODS, ORDER_STATUSES, DELIVERY_STATUSES, CATEGORIES, FLAT_CATEGORIES, N_CUSTOMERS, N_PRODUCTS, N_ORDERS, N_EVENTS    

In [0]:
fake = Faker("en_US") 

In [0]:
random.seed(42)
BASE_DIR = "/Volumes/e2e_databricks/raw/storage"
YEARS = [2022, 2023, 2024, 2025]

Helpers

In [0]:
def rand_date_in_year(year: int) -> datetime:
    start = datetime(year, 1, 1)
    end   = datetime(year, 12, 31, 23, 59, 59)
    return start + (end - start) * random.random()


def rand_date(start_days_ago=730, end_days_ago=0) -> datetime:
    start = datetime.now() - timedelta(days=start_days_ago)
    end   = datetime.now() - timedelta(days=end_days_ago)
    return start + (end - start) * random.random()


def dirty_timestamp(dt: datetime) -> str:
    fmt = random.choices(
        ["%Y-%m-%d %H:%M:%S", "%d/%m/%Y %H:%M", "%b %d %Y", "%Y-%m-%dT%H:%M:%SZ"],
        weights=[50, 20, 15, 15],
    )[0]
    return dt.strftime(fmt)


def dirty_price(price: float) -> str:
    variant = random.choices(["clean", "dollar", "comma", "string"], weights=[60, 15, 15, 10])[0]
    if variant == "clean":
        return str(round(price, 2))
    elif variant == "dollar":
        return f"${price:.2f}"
    elif variant == "comma":
        return str(price).replace(".", ",")
    else:
        return f"{price:.2f} USD"

def dirty_customer_id(cid: int) -> str:
    return random.choices([str(cid), f"CUST_{cid:06d}"], weights=[65, 35])[0]

def out_path(table: str, year: int) -> str:
    """Returns path and ensures the directory exists."""
    import os
    path = f"{BASE_DIR}/{table}"
    os.makedirs(path, exist_ok=True)
    return f"{path}/{table}_{year}.csv"

Table Generators

In [0]:
def generate_customers(year: int, n_per_year: int = None) -> pl.DataFrame:
    """
    Only customers created in `year`. Each customer appears in exactly one file.
    Dirty: ~3% dupes, ~5% null email, ~4% null phone,
           ~2% corrupted age, mixed timestamps.
    """
    if n_per_year is None:
        n_per_year = N_CUSTOMERS // len(YEARS)

    year_index = YEARS.index(year)
    start_idx = year_index * n_per_year
    year_ids = CUSTOMER_IDS[start_idx : start_idx + n_per_year]

    rows = []
    for cid in year_ids:
        created = rand_date_in_year(year)
        age = random.randint(18, 75)
        if random.random() < 0.02:
            age = random.choice([5, 0, 150, 999])

        f_name = fake.name()
        f_last_name = fake.last_name()
        email = f"{f_name}{f_last_name}@gmail.com".lower()

        rows.append({
            "customer_id": cid,
            "full_name": f"{f_name} {f_last_name}",
            "email": email if random.random() > 0.05 else None,
            "phone": fake.phone_number() if random.random() > 0.04 else None,
            "age": age,
            "gender": random.choice(["M", "F", "Other", None]),
            "country": fake.country(),
            "city": fake.city(),
            "created_at": dirty_timestamp(created),
            "is_active": random.choice([True, True, True, False]),
        })

    df = pl.DataFrame(rows)
    n_dupes = max(1, int(n_per_year * 0.03))
    df = pl.concat([df, df.sample(n_dupes, with_replacement=True)]).sample(fraction=1.0, shuffle=True)

    df.write_csv(out_path("customers", year))
    print(f"customers_{year}.csv — {len(df):,} rows  ({n_dupes} dupes)")
    return df

def generate_products() -> pl.DataFrame:
    """Dirty: ~5% null description, ~3% negative stock."""
    import os
    rows = []
    for pid in PRODUCT_IDS:
        cat, sub = random.choice(FLAT_CATEGORIES)
        price = round(random.uniform(5, 1500), 2)
        stock = random.randint(0, 500)
        if random.random() < 0.03:
            stock = random.randint(-50, -1)

        rows.append({
            "product_id": pid,
            "name": fake.catch_phrase().title()[:80],
            "category": cat,
            "subcategory": sub,
            "price_usd": price,
            "cost_usd": round(price * random.uniform(0.3, 0.7), 2),
            "stock": stock,
            "description": fake.text(100) if random.random() > 0.05 else None,
            "brand": fake.company()[:40],
            "created_at": rand_date(1000, 60).strftime("%Y-%m-%d"),
        })

    df = pl.DataFrame(rows)
    path = f"{BASE_DIR}/products"
    os.makedirs(path, exist_ok=True)
    df.write_csv(f"{path}/products.csv")
    print(f"products.csv — {len(df):,} rows")
    return df

def generate_orders(year: int, available_cust_ids: list, n_orders: int = None) -> pl.DataFrame:
    """
    `available_cust_ids` = all customers registered up to this year.
    Dirty: mixed timestamps, mixed customer_id format, null shipping,
           ~5% duplicate order_id, dirty prices, ~2% impossible dates.
    """
    if n_orders is None:
        n_orders = N_ORDERS // len(YEARS)

    rows = []
    for o in range(n_orders):
        cid = random.choice(available_cust_ids)
        order_dt = rand_date_in_year(year)
        ship_dt = order_dt + timedelta(days=random.randint(1, 7))
        status = random.choice(ORDER_STATUSES)

        if random.random() < 0.02:
            order_dt, ship_dt = ship_dt, order_dt

        rows.append({
            "order_id": o + 1,
            "customer_id": dirty_customer_id(cid),
            "channel": random.choice(CHANNELS),
            "status": status,
            "total_amount": dirty_price(round(random.uniform(10, 3000), 2)),
            "currency": random.choices(["USD", "MXN", "EUR"], weights=[70, 20, 10])[0],
            "shipping_address": fake.address().replace("\n", ", ") if random.random() > 0.04 else None,
            "order_date": dirty_timestamp(order_dt),
            "shipped_date": dirty_timestamp(ship_dt) if status != "cancelled" else None,
            "ingestion_time": datetime.now().isoformat()
        })

    df = pl.DataFrame(rows)
    n_dupes = max(1, int(n_orders * 0.05))
    dupes = df.sample(n_dupes, with_replacement=True).with_columns(
        pl.lit(datetime.now().isoformat()).alias("ingestion_time")
    )
    df = pl.concat([df, dupes]).sample(fraction=1.0, shuffle=True)

    df.write_csv(out_path("orders", year))
    print(f"  ✓ orders_{year}.csv — {len(df):,} rows  ({n_dupes} dupes)")
    return df

def generate_order_items(orders_df: pl.DataFrame, year: int) -> pl.DataFrame:
    """Dirty: ~3% null qty, ~2% qty=0."""
    rows = []
    for oid in orders_df["order_id"].unique().to_list():
        for x in range(random.randint(1, 5)):
            qty = random.randint(1, 10)
            if random.random() < 0.03:
                qty = None
            elif random.random() < 0.02:
                qty = 0
            rows.append({
                "item_id":      x + 1,
                "order_id":     oid,
                "product_id":   random.choice(PRODUCT_IDS),
                "quantity":     qty,
                "unit_price":   round(random.uniform(5, 1500), 2),
                "discount_pct": round(random.uniform(0, 0.4), 2) if random.random() > 0.6 else 0.0,
            })

    df = pl.DataFrame(rows)
    df.write_csv(out_path("order_items", year))
    print(f"  ✓ order_items_{year}.csv — {len(df):,} rows")
    return df


# ─────────────────────────────────────────────────────────────────────────
# PAYMENTS
# ─────────────────────────────────────────────────────────────────────────
def generate_payments(orders_df: pl.DataFrame, year: int) -> pl.DataFrame:
    """Dirty: ~5% double attempts, ~3% null method, ~2% partial amount."""
    rows = []
    for oid in orders_df["order_id"].unique().to_list():
        n_attempts = 2 if random.random() < 0.05 else 1
        for attempt in range(1, n_attempts + 1):
            amount = round(random.uniform(10, 3000), 2)
            if random.random() < 0.02:
                amount = round(amount * random.uniform(0.5, 0.99), 2)
            rows.append({
                "payment_id":     str(uuid.uuid4()),
                "order_id":       oid,
                "payment_method": random.choice(PAYMENT_METHODS) if random.random() > 0.03 else None,
                "amount": amount,
                "currency": "USD",
                "status": "failed" if attempt < n_attempts else random.choice(["approved", "approved", "pending", "chargeback"]),
                "attempt": attempt,
                "processed_at": rand_date_in_year(year).isoformat(),
                "gateway": random.choice(["stripe", "paypal", "conekta", "mercadopago"]),
            })

    df = pl.DataFrame(rows)
    df.write_csv(out_path("payments", year))
    print(f"  ✓ payments_{year}.csv — {len(df):,} rows")
    return df


# ─────────────────────────────────────────────────────────────────────────
# DELIVERIES
# ─────────────────────────────────────────────────────────────────────────
def generate_deliveries(orders_df: pl.DataFrame, year: int) -> pl.DataFrame:
    """Dirty: ~5% null carrier, ~4% null tracking, ~3% delivered before shipped."""
    shipped = orders_df.filter(
        pl.col("status").is_in(["completed", "disputed", "refunded"])
    )["order_id"].unique().to_list()

    rows = []
    for oid in shipped:
        ship_dt  = rand_date_in_year(year)
        deliv_dt = ship_dt + timedelta(days=random.randint(1, 14))
        if random.random() < 0.03:
            deliv_dt = ship_dt - timedelta(days=random.randint(1, 3))

        rows.append({
            "delivery_id":     str(uuid.uuid4()),
            "order_id":        oid,
            "carrier":         random.choice(["FedEx", "UPS", "DHL", "USPS", "Estafeta"]) if random.random() > 0.05 else None,
            "tracking_number": fake.bothify("??-########-??").upper() if random.random() > 0.04 else None,
            "status":          random.choice(DELIVERY_STATUSES),
            "shipped_at":      ship_dt.isoformat(),
            "delivered_at":    deliv_dt.isoformat(),
            "estimated_days":  random.randint(2, 10),
        })

    df = pl.DataFrame(rows)
    df.write_csv(out_path("deliveries", year))
    print(f"  ✓ deliveries_{year}.csv — {len(df):,} rows")
    return df

Main

In [0]:
print("\n🛒  E-Commerce Data Generator")
print("=" * 45)

# Products: dimension table, generated once (no year partition)
products = generate_products()

# Customers accumulate year over year:
# orders in 2024 can reference customers from 2022.
n_per_year         = N_CUSTOMERS // len(YEARS)
available_cust_ids = []

for year in YEARS:
    print(f"\n── {year} {'─' * 35}")

    customers_year = generate_customers(year, n_per_year)

    # Expand the customer pool with this year's new IDs
    year_index          = YEARS.index(year)
    start_idx           = year_index * n_per_year
    available_cust_ids += CUSTOMER_IDS[start_idx : start_idx + n_per_year]

    orders      = generate_orders(year, available_cust_ids)
    order_items = generate_order_items(orders, year)
    payments    = generate_payments(orders, year)
    deliveries  = generate_deliveries(orders, year)

print("""
\n✅  Done. Output structure:

  {BASE_DIR}/
  ├── products/products.csv
  ├── customers/year=YYYY/customers_YYYY.csv
  ├── orders/year=YYYY/month=MM/orders_YYYY_MM.csv
  ├── order_items/year=YYYY/month=MM/order_items_YYYY_MM.csv
  ├── payments/year=YYYY/payments_YYYY.csv
  └── deliveries/year=YYYY/deliveries_YYYY.csv
""")